# Recovery forecasts for UP5 bonds

The empirical study in Section 4 of the paper uses two fixed UP5 partitions with 1,293 training bonds and 432 evaluation bonds. For the target-only benchmarks, we use 1,034 training observations for fitting and reserve 259 for validation. The modeling extract omits individual default dates and issuer identifiers, so separation by time or issuer cannot be verified.

Select a mode in the first code cell, then choose **Run All**.

| Mode | Execution |
|---|---|
| `review` (default) | Displays the aggregate results supplied with the repository. Runs without private workbooks or model weights. |
| `rebuild` | Fits the target-only benchmarks and backbone comparisons from the four UP5 workbooks. TabPFN requires its pretrained weights. Each run writes to a new directory. |
| `historical` | Displays the aggregate results and reconstructs the earlier UP5 figures from the workbooks, including three series of 15 training runs. |

In [1]:
MODE = "review"  # review | rebuild | historical
from contextlib import redirect_stderr, redirect_stdout
from datetime import datetime
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

if MODE not in {"review", "rebuild", "historical"}:
    raise ValueError("Choose review, rebuild or historical.")
cwd = Path.cwd().resolve()
repo = next(p for p in (cwd, *cwd.parents) if (p / "src/experiments/up5").is_dir())
sys.path.insert(0, str(repo / "src"))
data = repo / "data/UP5"
tables = repo / "results/up5"
if MODE != "review":
    required = [f"{split}_{kind}2.xlsx" for split in ("train", "test") for kind in ("features", "labels")]
    missing = [name for name in required if not (data / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing UP5 workbooks: {missing}")
if MODE == "rebuild":
    import tabpfn
    from simulation.notebook_reporting import quiet_epochs
    tables = repo / "outputs/up5_rebuild" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    tables.mkdir(parents=True)

def fit_step(label, function, *args):
    print(f"{label} | running", flush=True)
    with (tables / "training.log").open("a") as log, redirect_stdout(log), redirect_stderr(log), quiet_epochs():
        result = function(*args)
    print(f"{label} | complete", flush=True)
    return result

def show(filename):
    table = pd.read_csv(tables / filename, index_col=0)
    table.index = table.index.to_series().replace({
        "FT-MDN": "FT–MDN–T", "FT-Transformer-MDN": "FT–MDN–T",
        "FT-pooled-MLP-MDN": "FT pooled-MLP–MDN"
    })
    display(table.round(4))

print(f"Mode: {MODE} | revision tables: {tables.relative_to(repo)}")

Mode: review | revision tables: results/up5


## 1. Target-only point forecasts

On the 317 encoded predictors, we compare the conditional mean from the FT–MDN–Transformer (FT–MDN–T) with the training mean, XGBoost, MLP and TabPFN. The table reports R² and mean absolute error (MAE) on the 432 evaluation bonds. Elapsed time refers to the machine used for the run.

In [2]:
if MODE == "rebuild":
    from experiments.up5.target_only import run_benchmark
    fit_step("Target-only point forecasts", run_benchmark, data, tables)
show("scores.csv")

,r2,mae,seconds,n_fit,n_validation
model,,,,,
Training mean,-0.0029,0.2967,0.0006,1034,259
XGBoost,0.5006,0.1478,0.6638,1034,259
MLP,0.4661,0.1817,1.4213,1034,259
FT–MDN–T,0.4108,0.1848,3174.7538,1034,259
TabPFN,0.5449,0.1630,799.7607,1034,259


## 2. Predictive distributions and interval coverage

We score the FT–MDN–T mixture and the Gaussian ridge forecast using CRPS and negative log likelihood (NLL). Lower scores indicate better distribution forecasts. The empirical distribution of training recoveries provides an unconditional CRPS benchmark. For the central 80% intervals, we report coverage and mean width. The second table compares the mean forecast probabilities of below-zero and above-par recovery with their observed frequencies.

In [3]:
if MODE == "rebuild":
    from experiments.up5.distribution import compare
    from experiments.up5.cohort_summary import tail_probabilities
    density = pd.DataFrame(fit_step("Density scores", compare, data, tables)).T
    density.to_csv(tables / "density-scores.csv", index_label="model")
    tail_probabilities(tables).to_csv(tables / "tails.csv", index_label="measure")
show("density-scores.csv")
show("tails.csv")

,CRPS,NLL,coverage80,width80
model,,,,
FT–MDN–T,0.1308,-0.2815,0.7222,0.4566
Gaussian ridge,0.1542,0.1352,0.7940,0.6714
Training empirical,0.1939,NaN,NaN,NaN


,Below zero (%),Above par (%)
measure,,
Mean forecast probability,1.0743,6.7688
Observed frequency,0.0000,10.8796


## 3. Backbone comparisons on encoded and native features

The first table compares FT–MDN–T with an MLP mixture-density model on the 317 encoded predictors. In the second comparison, both models use the same tokenizer design for 155 numerical and nine categorical fields, and the same mixture-density head design. One model applies self-attention across feature tokens. The other applies an MLP to each token and averages the resulting representations. The table includes parameter counts because the two backbones differ in capacity.

In [4]:
if MODE == "rebuild":
    from experiments.up5.encoded_backbones import compare_backbones
    from experiments.up5.native_backbones import compare_native
    fit_step("Encoded backbone comparison", compare_backbones, data, tables)
    native = fit_step("Native-token backbone comparison", compare_native, data, tables / "native")
    native.to_csv(tables / "native-backbones.csv")
show("mdn-backbones.csv")
show("native-backbones.csv")

,R2,MAE,CRPS,NLL,coverage80,width80
FT–MDN–T,0.4108,0.1848,0.1308,-0.2815,0.7222,0.4566
MLP-MDN,0.4774,0.1716,0.1243,-0.3377,0.7014,0.3959


,parameters,epoch,val_R2,R2,MAE,CRPS,NLL,coverage80,width80,seconds
model,,,,,,,,,,
FT–MDN–T,408631,59,0.4152,0.3983,0.1817,0.1334,-0.2704,0.6898,0.4012,909.2893
FT pooled-MLP–MDN,64623,73,0.3684,0.3236,0.2112,0.1483,-0.0572,0.6806,0.5297,108.1440


## 4. Historical UP5 figures and cohort summary

In `historical` mode, the final cell writes the UP5 panels to `outputs/up5_historical/`. The [figure code](../src/experiments/up5/historical_figures.py) uses the [archived UP5 notebook](archive/up5_mdn_ablations.ipynb). Cell numbers below count from zero and include Markdown cells.

| Paper figure | Analysis | Output |
|---|---|---|
| 4–5 · UP5 panels | Recovery histogram for the training bonds and correlations among the 31 shared numerical predictors. | `rr_dist_up5.png`, `correlation_heatmap_up5.png` |
| 6 · Individual bond recoveries | Conditional density, mixture components, predicted mean and observed recovery for evaluation rows 57, 58, 63 and 89 (zero-based, in panel order). Archived cell 16. | `mdn_dist_sample_1.pdf` through `mdn_dist_sample_4.pdf` |
| 7 · Portfolio recoveries | Training and evaluation histograms, each overlaid with the mean predictive density for that partition. Archived cell 10. | `up5_ft_mdn_density_side_by_side.pdf` |
| 8 · Categorical embeddings and dummy encoding | Validation R² over 15 seeds per encoding, with interquartile bands and early-stopping markers. Archived cells 95, 96 and 100. | `ft_mdn_transformer_cat_vs_dummy.pdf` |
| 9 · Within-UP5 schema expansion | Validation R² during shared-input pretraining and full-input fine-tuning. Archived cells 69 and 70. | `ft_mdn_transformer_transfer_up5_only_heterogeneous.pdf` |

The earlier fits used the supplied evaluation partition for validation. Figure 8 varies the input encoding. Section 3 of this notebook varies the backbone while holding the native tokenizer design fixed. The historical preprocessing processes overlapping event-type prefixes in their original order. The revised native tokenizer separates the two fields explicitly.

For Figure 9, the archived configuration selects 36 shared predictors and allows up to 80 pretraining epochs followed by 100 additional fine-tuning epochs. The manuscript describes 37 predictors and 100 pretraining epochs. The figure shows the two training phases. Verifying the caption’s comparison with scratch would require a separate scratch series. The original checkpoints are unavailable.

The manuscript supplies the GCD panels of Figures 4–5 and the cross-portfolio forecasts in Figure 10. Regeneration requires restricted GCD data and, for Figure 10, the original run records. The feature-overlap diagram in Figure 4 is drawn in LaTeX.

The table below gives the number of bonds, mean recovery, standard deviation and skewness for each UP5 partition and for the combined cohort.

In [5]:
if MODE == "rebuild":
    from experiments.up5.cohort_summary import recovery_summary
    recovery_summary(data).to_csv(tables / "recovery-summary.csv", index_label="UP5 cohort")
show("recovery-summary.csv")

,count,mean,std,skew
UP5 cohort,,,,
Training,1293.0,0.4835,0.3295,0.3379
Evaluation,432.0,0.4876,0.3388,0.3144
Combined,1725.0,0.4845,0.3318,0.3318


In [6]:
if MODE == "historical":
    folder = repo / "outputs/up5_historical"
    folder.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ, PYTHONPATH=str(repo / "src"), MPLBACKEND="Agg", PYTHONHASHSEED="0")
    print(f"Historical fits and figures | progress log: {folder.relative_to(repo)}/training.log", flush=True)
    with (folder / "training.log").open("w") as log:
        subprocess.run([sys.executable, "-u", "-m", "experiments.up5.historical_figures", str(repo), str(folder)],
                       cwd=repo, env=env, stdout=log, stderr=subprocess.STDOUT, check=True)
    for path in sorted(folder.iterdir()):
        print(path.relative_to(repo))